In [1]:
"""High-resolution Matplotlib chart exporters."""
from __future__ import annotations
from pathlib import Path
from typing import Any
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

def _save(fig, out:Path, name:str)->str:
    out.mkdir(parents=True,exist_ok=True); p=out/f"{name}.png"; fig.tight_layout(); fig.savefig(p,dpi=180,bbox_inches="tight"); plt.close(fig); return str(p)

def generate_charts(df:pd.DataFrame, analysis:dict, output_dir:str|Path)->dict:
    """Generate seven charts; unavailable subsets produce an explanatory chart."""
    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True); paths={}; x=df.copy()
    x["InvoiceDate"]=pd.to_datetime(x.get("InvoiceDate"),errors="coerce"); x["Revenue"]=pd.to_numeric(x.get("Quantity",0),errors="coerce")*pd.to_numeric(x.get("UnitPrice",0),errors="coerce")
    s=x[(x["Quantity"]>0)&~x["InvoiceNo"].astype(str).str.startswith("C")]
    months=s.set_index("InvoiceDate")["Revenue"].resample("MS").sum() if not s.empty else pd.Series(dtype=float)
    fig,ax=plt.subplots(figsize=(10,5)); months.plot(ax=ax,marker="o",title="Monthly Revenue Trend"); ax.set_ylabel("Revenue"); paths["monthly_trends"]=_save(fig,out,"monthly_trends")
    customers=s.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False) if "CustomerID" in s else pd.Series(dtype=float)
    fig,ax=plt.subplots(figsize=(8,5)); customers.cumsum().div(customers.sum()).mul(100).plot(ax=ax,title="Pareto Customer Revenue",marker="."); ax.set_ylabel("Cumulative revenue (%)"); paths["pareto"]=_save(fig,out,"pareto")
    fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(s.get("Quantity",[]),s.get("Revenue",[]),alpha=.5); ax.set(title="Anomaly Scatter Plot",xlabel="Quantity",ylabel="Revenue"); paths["anomalies"]=_save(fig,out,"anomalies")
    prod=s.groupby("Description")["Revenue"].sum().nlargest(10).sort_values(); fig,ax=plt.subplots(figsize=(8,5)); prod.plot.barh(ax=ax,title="Top Products"); paths["top_products"]=_save(fig,out,"top_products")
    countries=s.groupby("Country")["Revenue"].sum().nlargest(10).sort_values() if "Country" in s else pd.Series(dtype=float); fig,ax=plt.subplots(figsize=(8,5)); countries.plot.barh(ax=ax,title="Top Countries"); paths["top_countries"]=_save(fig,out,"top_countries")
    fig,ax=plt.subplots(figsize=(8,5)); customers.plot.hist(ax=ax,bins=20,title="Customer Revenue Distribution"); ax.set_xlabel("Revenue"); paths["customer_distribution"]=_save(fig,out,"customer_distribution")
    daily=s.set_index("InvoiceDate")["Revenue"].resample("D").sum() if not s.empty else pd.Series(dtype=float); fig,ax=plt.subplots(figsize=(9,4)); daily.plot(ax=ax,title="Daily Revenue / Order Activity"); paths["order_metrics"]=_save(fig,out,"order_metrics")
    return paths
